In [ ]:
# Import necessary libraries
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import SparkSession

# Initialize Spark Session if not already running
spark = SparkSession.builder \
    .appName("MovieRecommendationModels") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Load training data
train_df = spark.read.parquet("train_data")
test_df = spark.read.parquet("test_data")

# Build the ALS recommendation model
print("Training ALS model...")
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=False
)

# Train the model
als_model = als.fit(train_df)

# Make predictions on test data
predictions = als_model.transform(test_df)

# Evaluate the model
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error = {rmse}")

# Generate top 10 movie recommendations for a subset of users
users = train_df.select("userId").distinct().limit(3)
user_recs = als_model.recommendForUserSubset(users, 10)

# Show recommendations
print("Sample recommendations:")
user_recs.show(truncate=False)

# Save model for later use
als_model.save("als_model")
print("Collaborative filtering model saved!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/17 00:09:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/17 00:09:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Training ALS model...


25/04/17 00:09:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/04/17 00:09:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


Root-mean-square error = 0.8816542515385917
Sample recommendations:
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                                 |
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|474   |[{4495, 4.8834805}, {2295, 4.7330494}, {8235, 4.718793}, {7096, 4.667042}, {6791, 4.661201}, {7121, 4.61134}, {43376, 4.6054955}, {7767, 4.6014333}, {33649, 4.5964756}, {3200, 4.5789137}]     |
|26    |[{96004, 4.297183}, {25771, 4.2594457}, {26133, 4.156501}, {171495, 4.1467752}, {3266, 4.127818}, {928, 4.0899105}, 

25/04/17 00:09:27 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
